# 📖 Notebook 1: Latency Numbers Every Programmer Should Know

Understanding latency — how long operations take — is the **foundation** of system design.
If you don't know that a disk read is 10,000× slower than a memory read, you'll make terrible design decisions.

## Learning Objectives

By the end of this notebook, you'll be able to:
- Recite the latency hierarchy from L1 cache to cross-continent network
- **Measure actual latencies** on your own machine with Python
- Spot BAD designs that ignore latency and fix them with latency-aware patterns
- Explain why caching, batching, and locality matter

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 01-foundations/numbers-to-know
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import time
import timeit
import os
import json
import tempfile

# Database connection settings
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "numbers_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker compose up -d")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")
    print("   Run: docker compose up -d")

## 📊 The Famous Latency Hierarchy

Jeff Dean (Google) originally published "Latency Numbers Every Programmer Should Know."
These numbers have been updated over the years as hardware improves. Here are the **2026 numbers**:

```
┌──────────────────────────────────────────────────────────────────┐
│  OPERATION                           LATENCY        COMPARISON  │
├──────────────────────────────────────────────────────────────────┤
│  L1 cache reference ............... 1 ns            1 sec       │
│  L2 cache reference ............... 4 ns            4 sec       │
│  L3 cache reference ............... 10 ns           10 sec      │
│  Main memory (RAM) reference ...... 100 ns          1.5 min     │
│  Compress 1KB with Snappy ......... 2,000 ns        30 min      │
│  Read 1 MB from RAM ............... 250,000 ns      3 days      │
│  SSD random read .................. 16,000 ns       4.5 hours   │
│  Read 1 MB from SSD ............... 1,000,000 ns    11.5 days   │
│  Round trip in datacenter ......... 500,000 ns      5.8 days    │
│  HDD disk seek .................... 2,000,000 ns    23 days     │
│  Read 1 MB from HDD ............... 5,000,000 ns    58 days     │
│  Internet CA → Netherlands → CA ... 150,000,000 ns  4.7 years   │
└──────────────────────────────────────────────────────────────────┘

  The "Comparison" column scales everything so L1 cache = 1 second.
  If an L1 cache hit took 1 second, a disk seek would take 23 DAYS.
```

💡 **The key insight**: There's a **10,000× difference** between memory and disk.
That's why caching exists. That's why databases use buffer pools. That's why SSDs changed everything.

In [ ]:
# Let's print this as a formatted table we can reference

latency_numbers = [
    ("L1 cache reference",              "1 ns",         1),
    ("L2 cache reference",              "4 ns",         4),
    ("L3 cache reference",              "10 ns",        10),
    ("Mutex lock/unlock",               "17 ns",        17),
    ("Main memory (RAM) reference",     "100 ns",       100),
    ("Compress 1KB (Snappy)",           "2 μs",         2_000),
    ("Read 1 MB from RAM",             "250 μs",       250_000),
    ("SSD random read (NVMe)",          "16 μs",        16_000),
    ("Read 1 MB from SSD",             "1 ms",         1_000_000),
    ("Round trip in datacenter",        "500 μs",       500_000),
    ("Redis GET (localhost)",           "0.1-0.5 ms",   300_000),
    ("PostgreSQL simple query",         "1-5 ms",       3_000_000),
    ("HDD disk seek",                   "2 ms",         2_000_000),
    ("Read 1 MB from HDD",            "5 ms",         5_000_000),
    ("Internet round trip (same region)", "50 ms",      50_000_000),
    ("Internet CA → EU → CA",          "150 ms",       150_000_000),
]

print("📊 Latency Numbers Every Programmer Should Know (2026)")
print("=" * 70)
print(f"{'Operation':<38} {'Latency':>12}  {'vs L1 cache':>12}")
print("-" * 70)

for name, latency_str, ns in latency_numbers:
    multiplier = f"{ns:,}×" if ns > 1 else "1×"
    print(f"  {name:<36} {latency_str:>12}  {multiplier:>12}")

print()
print("💡 Memory is ~100× slower than L1 cache.")
print("   Disk is ~100× slower than memory.")
print("   Network is ~100× slower than local disk.")
print("   Each layer adds roughly 2 orders of magnitude!")

## 🔬 Let's Measure It Ourselves!

Theory is nice, but let's **actually measure** these latencies on our machine.
We can't measure L1/L2 cache from Python (the interpreter adds overhead), but we CAN measure
everything from memory access up to network calls.

We'll use Python's `timeit` module for precise measurements.

In [ ]:
# ── Benchmark 1: CPU Arithmetic (represents "fast" baseline) ──

def bench_arithmetic():
    """Simple addition — as fast as Python can go."""
    x = 1 + 1
    return x

n_runs = 1_000_000
cpu_time = timeit.timeit(bench_arithmetic, number=n_runs)
cpu_per_op_ns = (cpu_time / n_runs) * 1e9

print("⚡ Benchmark 1: CPU Arithmetic (1 + 1)")
print(f"   Ran {n_runs:,} times")
print(f"   Average: {cpu_per_op_ns:.0f} ns per operation")
print(f"   (This includes Python interpreter overhead — real CPU ops are ~1 ns)")
print()

# ── Benchmark 2: Python Dict Lookup (represents memory/cache access) ──

big_dict = {f"key:{i}": f"value_{i}" for i in range(10_000)}

def bench_dict_lookup():
    """Dictionary lookup — Python's hash table, all in RAM."""
    return big_dict["key:5000"]

dict_time = timeit.timeit(bench_dict_lookup, number=n_runs)
dict_per_op_ns = (dict_time / n_runs) * 1e9

print("🧠 Benchmark 2: Python Dict Lookup (RAM)")
print(f"   Ran {n_runs:,} times")
print(f"   Average: {dict_per_op_ns:.0f} ns per operation")
print()

# ── Benchmark 3: List Index Access (represents sequential memory) ──

big_list = list(range(10_000))

def bench_list_access():
    """List index access — sequential memory."""
    return big_list[5000]

list_time = timeit.timeit(bench_list_access, number=n_runs)
list_per_op_ns = (list_time / n_runs) * 1e9

print("📋 Benchmark 3: List Index Access (RAM)")
print(f"   Ran {n_runs:,} times")
print(f"   Average: {list_per_op_ns:.0f} ns per operation")

In [ ]:
# ── Benchmark 4: File Read from SSD ──

# Create a temp file with some data
tmp_file = os.path.join(tempfile.gettempdir(), "bench_test.txt")
with open(tmp_file, "w") as f:
    f.write("Hello, this is a benchmark test! " * 30)  # ~1 KB

def bench_file_read():
    """Open and read a small file from disk (SSD)."""
    with open(tmp_file, "r") as f:
        return f.read()

n_file = 10_000
file_time = timeit.timeit(bench_file_read, number=n_file)
file_per_op_us = (file_time / n_file) * 1e6

print("💾 Benchmark 4: File Read from SSD (~1 KB)")
print(f"   Ran {n_file:,} times")
print(f"   Average: {file_per_op_us:.1f} μs per operation")
print(f"   ({file_per_op_us / 1000:.3f} ms)")
print()

# ── Benchmark 5: Redis GET ──

r = get_redis_client()
r.set("bench:key", "Hello from Redis! " * 5)

def bench_redis_get():
    """Redis GET over localhost network."""
    return r.get("bench:key")

n_redis = 10_000
redis_time = timeit.timeit(bench_redis_get, number=n_redis)
redis_per_op_us = (redis_time / n_redis) * 1e6

print("⚡ Benchmark 5: Redis GET (localhost)")
print(f"   Ran {n_redis:,} times")
print(f"   Average: {redis_per_op_us:.1f} μs per operation")
print(f"   ({redis_per_op_us / 1000:.3f} ms)")
print()

# ── Benchmark 6: PostgreSQL Simple Query ──

def bench_postgres_query():
    """PostgreSQL simple SELECT over localhost network."""
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute("SELECT value FROM benchmark_kv WHERE key = 'key:5000'")
    result = cur.fetchone()
    conn.close()
    return result

n_pg = 1_000
pg_time = timeit.timeit(bench_postgres_query, number=n_pg)
pg_per_op_us = (pg_time / n_pg) * 1e6

print("🐘 Benchmark 6: PostgreSQL Simple Query (localhost)")
print(f"   Ran {n_pg:,} times")
print(f"   Average: {pg_per_op_us:.1f} μs per operation")
print(f"   ({pg_per_op_us / 1000:.3f} ms)")

# Clean up
os.remove(tmp_file)

In [ ]:
# ── Benchmark 7: PostgreSQL with Connection Reuse ──
# Real apps reuse connections. Let's see how much faster it is.

conn = get_db_connection()

def bench_postgres_reuse():
    """PostgreSQL query with a reused connection."""
    cur = conn.cursor()
    cur.execute("SELECT value FROM benchmark_kv WHERE key = 'key:5000'")
    return cur.fetchone()

n_pg_reuse = 5_000
pg_reuse_time = timeit.timeit(bench_postgres_reuse, number=n_pg_reuse)
pg_reuse_per_op_us = (pg_reuse_time / n_pg_reuse) * 1e6
conn.close()

print("🐘 Benchmark 7: PostgreSQL with Connection Reuse")
print(f"   Ran {n_pg_reuse:,} times")
print(f"   Average: {pg_reuse_per_op_us:.1f} μs per operation")
print(f"   ({pg_reuse_per_op_us / 1000:.3f} ms)")
print()
print(f"💡 Connection reuse is {pg_per_op_us / pg_reuse_per_op_us:.1f}× faster!")
print(f"   This is why connection pools exist (pgbouncer, SQLAlchemy pool, etc.)")

## 📊 Visual Comparison

Let's plot all our measurements to really see the magnitude of differences.

In [ ]:
import matplotlib.pyplot as plt

# Collect all measurements (in microseconds for a common unit)
measurements = {
    "CPU\narithmetic":      cpu_per_op_ns / 1000,
    "Dict\nlookup":         dict_per_op_ns / 1000,
    "List\naccess":         list_per_op_ns / 1000,
    "File read\n(SSD)":     file_per_op_us,
    "Redis\nGET":           redis_per_op_us,
    "Postgres\n(new conn)": pg_per_op_us,
    "Postgres\n(reuse)":    pg_reuse_per_op_us,
}

labels = list(measurements.keys())
values = list(measurements.values())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Log scale (lets us see all values clearly across many orders of magnitude)
colors = ['#2ecc71', '#2ecc71', '#2ecc71', '#3498db', '#e67e22', '#e74c3c', '#e67e22']
bars = ax1.bar(labels, values, color=colors, edgecolor='white', linewidth=0.5)
ax1.set_yscale('log')
ax1.set_ylabel('Latency (us) - log scale')
ax1.set_title('Measured Latencies (Log Scale)')
ax1.grid(axis='y', alpha=0.3)

# Annotate each bar with its value in friendly units
for bar, val in zip(bars, values):
    label = f"{val:.0f} us" if val >= 1 else f"{val*1000:.0f} ns"
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
             label, ha='center', va='bottom', fontsize=8, fontweight='bold')

# Right: Linear scale (visually shows how massive the network gap is)
ax2.bar(labels, values, color=colors, edgecolor='white', linewidth=0.5)
ax2.set_ylabel('Latency (us) - linear scale')
ax2.set_title('Measured Latencies (Linear Scale)')
ax2.grid(axis='y', alpha=0.3)

plt.suptitle('Latency Benchmarks on YOUR Machine', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nThe log scale (left) lets you see all values.")
print("The linear scale (right) shows how MASSIVE the gap really is.")
print("Network operations completely dwarf in-memory operations!")


## ❌ BAD: Ignoring Latency (The N+1 Query Problem)

Here's a common mistake: fetching data one item at a time in a loop.
Each iteration pays the full network + database round trip cost.

**Scenario**: Display a feed of 20 posts with their authors.

In [ ]:
# ❌ BAD: Fetch each item separately (N+1 queries)
# This makes 1 query for posts + 20 queries for authors = 21 round trips!

conn = get_db_connection()
cur = conn.cursor()

start = time.time()

# Step 1: Get 20 events
cur.execute("SELECT id, user_id, event_type, payload FROM events LIMIT 20")
events = cur.fetchall()

# Step 2: For EACH event, fetch the user (N additional queries!)
results = []
for event in events:
    event_id, user_id, event_type, payload = event
    cur.execute("SELECT username, email FROM users WHERE id = %s", (user_id,))
    user = cur.fetchone()
    results.append({"event": event_type, "user": user[0] if user else "unknown"})

bad_time = (time.time() - start) * 1000
conn.close()

print("❌ BAD: N+1 Query Pattern")
print(f"   Queries executed: 1 + {len(events)} = {1 + len(events)} round trips")
print(f"   Total time: {bad_time:.2f} ms")
print(f"   Results: {len(results)} items")
print()
print("   Why is this bad?")
print(f"   Each query takes ~{pg_reuse_per_op_us/1000:.1f} ms (network + DB)")
print(f"   21 queries × {pg_reuse_per_op_us/1000:.1f} ms = {21 * pg_reuse_per_op_us/1000:.1f} ms of JUST network overhead!")
print("   With 1000 items, this becomes devastating.")

## ✅ GOOD: Latency-Aware Design (Batch + Cache)

Two simple fixes that respect latency:
1. **JOIN** — get all data in one query (1 round trip instead of N+1)
2. **Cache** — store hot data in Redis to skip the database entirely

In [ ]:
# ✅ GOOD (Fix 1): Use a JOIN — one query instead of 21

conn = get_db_connection()
cur = conn.cursor()

start = time.time()

cur.execute("""
    SELECT e.id, e.event_type, e.payload, u.username, u.email
    FROM events e
    JOIN users u ON e.user_id = u.id
    LIMIT 20
""")
results_join = cur.fetchall()

join_time = (time.time() - start) * 1000
conn.close()

print("✅ GOOD (Fix 1): Single JOIN Query")
print(f"   Queries executed: 1 round trip")
print(f"   Total time: {join_time:.2f} ms")
print(f"   Results: {len(results_join)} items")
print(f"   Speedup: {bad_time / join_time:.1f}× faster than N+1!")
print()

# ✅ GOOD (Fix 2): Cache the result in Redis

r = get_redis_client()

# Warm the cache (simulate app startup or first request)
r.set("feed:latest_20", json.dumps([
    {"event": row[1], "user": row[3]} for row in results_join
]), ex=60)  # expires in 60 seconds

start = time.time()
cached = json.loads(r.get("feed:latest_20"))
cache_time = (time.time() - start) * 1000

print("✅ GOOD (Fix 2): Read from Redis Cache")
print(f"   Queries executed: 0 DB queries, 1 Redis GET")
print(f"   Total time: {cache_time:.2f} ms")
print(f"   Results: {len(cached)} items")
print(f"   Speedup: {bad_time / cache_time:.1f}× faster than N+1!")
print()
print("=" * 55)
print(f"   ❌ BAD  (N+1 queries):   {bad_time:>8.2f} ms")
print(f"   ✅ JOIN (1 query):        {join_time:>8.2f} ms")
print(f"   ✅ Redis cache:           {cache_time:>8.2f} ms")
print("=" * 55)
print()
print("💡 Knowing the latency numbers tells you WHERE the time goes")
print("   and HOW to fix it: reduce round trips, use faster storage.")

## ⏱️ Tail Latency: Why p99 Matters More Than the Average

Averages lie. A service can have an average latency of 5 ms but still feel **slow** to users
because of "tail latency" — the slowest requests.

- **p50 (median)**: half of requests are faster than this, half are slower
- **p95**: 95% of requests are faster than this; 5% are slower
- **p99**: 99% of requests are faster than this; 1% are slower

In a system that makes 100 internal calls per page (microservices, fan-out), the **p99**
of one service becomes the **average** experience for the user. That's why companies like
Amazon optimize for p99/p99.9, not the mean.

Let's measure this on our Postgres queries:


In [ ]:
# Measure latency distribution (not just the average)
import statistics

conn = get_db_connection()

samples_us = []
for _ in range(2_000):
    cur = conn.cursor()
    t0 = time.perf_counter()
    cur.execute("SELECT value FROM benchmark_kv WHERE key = 'key:5000'")
    cur.fetchone()
    samples_us.append((time.perf_counter() - t0) * 1e6)
conn.close()

samples_us.sort()
def pct(p):
    return samples_us[int(len(samples_us) * p) - 1]

print("Latency distribution for 2,000 PostgreSQL SELECT queries")
print("=" * 55)
print(f"  min  : {min(samples_us):>8.2f} us")
print(f"  p50  : {pct(0.50):>8.2f} us  (median)")
print(f"  avg  : {statistics.mean(samples_us):>8.2f} us")
print(f"  p95  : {pct(0.95):>8.2f} us")
print(f"  p99  : {pct(0.99):>8.2f} us  <-- the slow 1%")
print(f"  max  : {max(samples_us):>8.2f} us")
print()
print(f"  p99 is ~{pct(0.99) / pct(0.50):.1f}x slower than the median!")
print()
print("Real-world implications:")
print("  - SLOs are usually defined on p99 (e.g. 'p99 < 100ms')")
print("  - Garbage collection, lock contention, and network jitter all show up here")
print("  - In a fan-out system, the slowest backend dominates the response time")


## 🧹 Cleanup

In [ ]:
r = get_redis_client()
keys = r.keys("bench:*") + r.keys("feed:*")
if keys:
    r.delete(*keys)
    print(f"Cleaned up {len(keys)} Redis keys")
else:
    print("Nothing to clean up")


## 📚 Summary

### Key Takeaways

1. **Latency spans 8 orders of magnitude** — from 1 ns (L1 cache) to 150 ms (cross-continent)
2. **Memory is ~100× faster than SSD**, SSD is ~100× faster than network
3. **N+1 queries are the #1 latency killer** — every round trip costs ~1-5 ms
4. **Batch operations and caching** turn 21 round trips into 1 (or zero)
5. **Connection reuse** alone can give you a 2-5× speedup

### Rules of Thumb for Interviews

- If it's in **memory** → microseconds (μs)
- If it's on **local SSD** → low milliseconds (ms)
- If it's across the **network** → milliseconds to tens of ms
- If it's **cross-region** → 50-150 ms

### Next Up

In **Notebook 2**, we'll use these latency numbers to calculate **throughput and capacity** —
how many requests per second can a system handle, and how much storage does it need?